In [1]:
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path
import pyarrow as pa, gc
import sys
import time

import os
import warnings
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

import xgboost as xgb
from lightgbm import LGBMRanker, early_stopping, log_evaluation
from catboost import CatBoostRanker, Pool


import torch
from torch.utils.data import Dataset

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
trainPath = Path("/kaggle/input/aeroclub-recsys-2025/train.parquet")
testPath = Path("/kaggle/input/aeroclub-recsys-2025/test.parquet")

# Now Create a class that process a dataframe

In [3]:
class PLDFPorcessor(): 
    def __init__(self, dfPath): 
        self.trainDFPl = pl.read_parquet(dfPath)
        self.trainDFPl = self.trainDFPl.drop([
            #"companyID",
            "profileId",
            "requestDate",
            #"nationality",
            "__index_level_0__",
            "miniRules1_percentage",
            "miniRules0_percentage",
            # dropping segment3 and segment2 columns as before
            "legs1_segments3_seatsAvailable",
            "legs1_segments3_flightNumber",
            "legs1_segments3_duration",
            "legs1_segments3_departureFrom_airport_iata",
            "legs1_segments3_cabinClass",
            "legs1_segments3_baggageAllowance_weightMeasurementType",
            "legs1_segments3_baggageAllowance_quantity",
            "legs1_segments3_arrivalTo_airport_iata",
            "legs1_segments3_arrivalTo_airport_city_iata",
            "legs1_segments2_seatsAvailable",
            "legs1_segments2_flightNumber",
            "legs1_segments2_duration",
            "legs1_segments2_departureFrom_airport_iata",
            "legs1_segments2_cabinClass",
            "legs1_segments2_baggageAllowance_weightMeasurementType",
            "legs1_segments2_baggageAllowance_quantity",
            "legs1_segments2_arrivalTo_airport_iata",
            "legs1_segments2_arrivalTo_airport_city_iata",
            "legs1_segments1_seatsAvailable",
            "legs1_segments1_flightNumber",
            "legs1_segments1_duration",
            "legs1_segments1_departureFrom_airport_iata",
            "legs1_segments1_cabinClass",
            "legs1_segments1_baggageAllowance_weightMeasurementType",
            "legs1_segments1_baggageAllowance_quantity",
            "legs1_segments1_arrivalTo_airport_iata",
            "legs1_segments1_arrivalTo_airport_city_iata",
            "legs0_segments3_seatsAvailable",
            "legs0_segments3_flightNumber",
            "legs0_segments3_duration",
            "legs0_segments3_departureFrom_airport_iata",
            "legs0_segments3_cabinClass",
            "legs0_segments3_baggageAllowance_weightMeasurementType",
            "legs0_segments3_baggageAllowance_quantity",
            "legs0_segments3_arrivalTo_airport_iata",
            "legs0_segments3_arrivalTo_airport_city_iata",
            "legs0_segments2_seatsAvailable",
            "legs0_segments2_flightNumber",
            "legs0_segments2_duration",
            "legs0_segments2_departureFrom_airport_iata",
            "legs0_segments2_cabinClass",
            "legs0_segments2_baggageAllowance_weightMeasurementType",
            "legs0_segments2_baggageAllowance_quantity",
            "legs0_segments2_arrivalTo_airport_iata",
            "legs0_segments2_arrivalTo_airport_city_iata",
            "legs0_segments1_seatsAvailable",
            "legs0_segments1_flightNumber",
            "legs0_segments1_duration",
            "legs0_segments1_departureFrom_airport_iata",
            "legs0_segments1_cabinClass",
            "legs0_segments1_baggageAllowance_weightMeasurementType",
            "legs0_segments1_baggageAllowance_quantity",
            "legs0_segments1_arrivalTo_airport_iata",
            "legs0_segments1_arrivalTo_airport_city_iata",
            # "corporateTariffCode",
            # These low-level iata columns dropped (if you want, keep for geo features)
            "legs0_segments0_arrivalTo_airport_city_iata", 
            "legs0_segments0_arrivalTo_airport_iata", 
            "legs0_segments0_departureFrom_airport_iata", 
            "legs0_segments0_duration", 
            "legs0_segments0_flightNumber", 
            "legs1_segments0_arrivalTo_airport_city_iata", 
            "legs1_segments0_arrivalTo_airport_iata", 
            "legs1_segments0_departureFrom_airport_iata", 
            "legs1_segments0_duration", 
            "legs1_segments0_flightNumber",
        ])
        pa.default_memory_pool().release_unused()
        gc.collect()

    def add_price_rank_features(self):
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("totalPrice").rank("dense").over("ranker_id").alias("price_rank"),
            pl.count().over("ranker_id").alias("group_count"),
            pl.col("legs0_duration").rank("dense").over("ranker_id").alias("duration_rank"),
        ])
        self.trainDFPl = self.trainDFPl.with_columns([
            ((pl.col("price_rank") - 1) / (pl.col("group_count") - 1).cast(pl.Float64)).alias("price_pct_rank")
        ])
        self.trainDFPl = self.trainDFPl.drop("group_count")
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") + 1).log().alias("log_price")
        ])

    def add_min_stopage_per_trip(self): 
        # Reusing your logic, renamed to snake_case to be consistent
        freq_split = self.trainDFPl['frequentFlyer'].fill_null("").str.split("/")
        codeCols = [
            "legs0_segments0_marketingCarrier_code",
            "legs0_segments1_marketingCarrier_code",
            "legs0_segments2_marketingCarrier_code",
            "legs0_segments3_marketingCarrier_code",
            "legs1_segments0_marketingCarrier_code",
            "legs1_segments1_marketingCarrier_code",
            "legs1_segments2_marketingCarrier_code",
            "legs1_segments3_marketingCarrier_code",
        ]
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.sum_horizontal(pl.col(col).is_not_null().cast(pl.UInt8) for col in codeCols).alias("seg_tempor"), 
            pl.col("corporateTariffCode").is_not_null().cast(pl.Int32).alias("has_corporate_tariff")
        ])
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("seg_tempor") == pl.col("seg_tempor").min().over("ranker_id")).cast(pl.Int32).alias("minStopagePerRankderID")
        ])
        self.trainDFPl = self.trainDFPl.drop("seg_tempor")

        self.trainDFPl = self.trainDFPl.with_columns([
            ((pl.col("miniRules1_monetaryAmount") == 0) & (pl.col("miniRules1_statusInfos") == 1)).cast(pl.Int8).alias("free_exchange")
        ])

        self.trainDFPl = self.trainDFPl.with_columns([
            freq_split.alias("freq_split"),  
            pl.sum_horizontal([pl.col(col).is_in(freq_split).cast(pl.Int32)for col in codeCols]).alias("frequentFlyer_marketingCarrier_match")
        ])

        self.trainDFPl = self.trainDFPl.drop("freq_split")

        del codeCols, freq_split
        pa.default_memory_pool().release_unused()
        gc.collect()

    def ff_flyer_bin_converter(self): 
        freq_split = self.trainDFPl['frequentFlyer'].fill_null("").str.split("/")

        codeCols = [
            "legs1_segments3_operatingCarrier_code",
            "legs1_segments2_operatingCarrier_code",
            "legs1_segments1_operatingCarrier_code",
            "legs1_segments0_operatingCarrier_code",
            "legs0_segments3_operatingCarrier_code",
            "legs0_segments2_operatingCarrier_code",
            "legs0_segments1_operatingCarrier_code",
            "legs0_segments0_operatingCarrier_code",
            "legs0_segments0_marketingCarrier_code",
            "legs0_segments1_marketingCarrier_code",
            "legs0_segments2_marketingCarrier_code",
            "legs0_segments3_marketingCarrier_code",
            "legs1_segments0_marketingCarrier_code",
            "legs1_segments1_marketingCarrier_code",
            "legs1_segments2_marketingCarrier_code",
            "legs1_segments3_marketingCarrier_code",
        ]
        
        self.trainDFPl = self.trainDFPl.with_columns(freq_split.alias("freq_split"))
        
        matches_exprs = [
            pl.when(pl.col(col).is_in(pl.col("freq_split"))).then(1).otherwise(0).alias(f"match_{col}")
            for col in codeCols
        ]
        
        self.trainDFPl = self.trainDFPl.with_columns(matches_exprs)
        
        match_cols = [f"match_{col}" for col in codeCols]
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.sum_horizontal(match_cols).alias("numberof_same_frequentFlyter_operator"),
            pl.col("bySelf").cast(pl.Int8).alias("bySelf")
        )
        
        drop_cols = codeCols + ["frequentFlyer"] + match_cols + ["freq_split"]
        self.trainDFPl = self.trainDFPl.drop(drop_cols)
        
        del codeCols, freq_split, matches_exprs, match_cols, drop_cols
        pa.default_memory_pool().release_unused()
        gc.collect()

        # Convert binary features to int8 safely
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("isAccess3D").fill_null(False).cast(pl.Int8),
            pl.col("isVip").fill_null(False).cast(pl.Int8),
            pl.col("sex").fill_null(False).cast(pl.Int8),
            pl.col("has_corporate_tariff").fill_null(False).cast(pl.Int8),
        ])
        
        # Aircraft code presence binary
        aircraft_cols = [
            "legs0_segments0_aircraft_code",
            "legs0_segments1_aircraft_code",
            "legs0_segments2_aircraft_code",
            "legs0_segments3_aircraft_code",
            "legs1_segments0_aircraft_code",
            "legs1_segments1_aircraft_code",
            "legs1_segments2_aircraft_code",
            "legs1_segments3_aircraft_code",
        ]
        self.trainDFPl = self.trainDFPl.with_columns(
            [pl.col(c).is_not_null().cast(pl.Int8).alias(c) for c in aircraft_cols]
        )
        
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.sum_horizontal(aircraft_cols).alias("total_travel_stopage")
        )
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.col("total_travel_stopage").min().over("ranker_id").alias("minimum_travel_segment"),
        )
        self.trainDFPl = self.trainDFPl.drop(aircraft_cols)
        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done ff_flyer_bin_converter!")




    def hhmmss_to_minutes(self, col_name) -> pl.Expr:
        def parse_duration(x):
            if len(x) < 3:
                return None
            first_part = x[0]
            minutes = float(x[1])
            seconds = float(x[2]) if len(x) > 2 else 0
            if '.' in first_part:
                day_str, hour_str = first_part.split('.')
                days = int(day_str)
                hours = int(hour_str)
            else:
                days = 0
                hours = int(first_part)
            total_minutes = days * 24 * 60 + hours * 60 + minutes + seconds / 60
            return total_minutes
    
        return (
            pl.col(col_name)
            .fill_null("00:00:00")
            .str.split(":")
            .map_elements(parse_duration, return_dtype=pl.Float64)
            .alias(col_name)
        )
    
    def hour_min_and_tax_converter(self): 
        self.trainDFPl = self.trainDFPl.with_columns([
            self.hhmmss_to_minutes("legs0_duration"),
            self.hhmmss_to_minutes("legs1_duration"),
        ])
        
        self.trainDFPl = self.trainDFPl.with_columns(
            (pl.col("legs0_duration") + pl.col("legs1_duration")).alias("total_travel_time_in_minutes")
        )
    
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("searchRoute").str.len_chars() // 3).alias("searchRoute"),
            (pl.col("taxes") / pl.col("totalPrice")).alias("tax_percentage"),
        ])

        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("legs0_duration").min().rank("dense").over("ranker_id").alias("legs0_duration_rank"),
            pl.col("legs1_duration").min().rank("dense").over("ranker_id").alias("legs1_duration_rank"),
            pl.col("total_travel_time_in_minutes").rank("dense").over("ranker_id").alias("total_travel_time_in_minutes_duration_rank"),
            
            #pl.col("total_travel_time_in_minutes").max().over("ranker_id").alias("total_travel_time_in_minutes_duration_rank_max"),
            #pl.col("total_travel_time_in_minutes").mean().over("ranker_id").alias("total_travel_time_in_minutes_duration_rank_mean"),
            #pl.col("total_travel_time_in_minutes").std().over("ranker_id").alias("total_travel_time_in_minutes_duration_rank_std"),
        ])
        gc.collect()
        print("Done hour_min_and_tax_converter")

    def others_and_duration(self):
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("taxes") / pl.col("pricingInfo_passengerCount")).alias("taxes"),
            (pl.col("totalPrice")/pl.col("pricingInfo_passengerCount")).alias("totalPrice")
        ])

        min_vals = (
            self.trainDFPl
            .group_by("ranker_id") 
            .agg([
                pl.col("totalPrice").min().alias("min_totalPrice"),
                pl.col("legs0_duration").min().alias("min_legs0_duration"),
                pl.col("legs1_duration").min().alias("min_legs1_duration"),
                pl.col("total_travel_time_in_minutes").min().alias("min_total_travel_time"),
            ])
        )
        
        self.trainDFPl = self.trainDFPl.join(min_vals, on="ranker_id", how="left")
        
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") - pl.col("min_totalPrice")).alias("totalPrice_diff"),
            (pl.col("legs0_duration") - pl.col("min_legs0_duration")).alias("legs0Duration_diff"),
            (pl.col("legs1_duration") - pl.col("min_legs1_duration")).alias("legs1Duration_diff"),
            (pl.col("total_travel_time_in_minutes") - pl.col("min_total_travel_time")).alias("totalDuration_diff"),
        ])
        
        self.trainDFPl = self.trainDFPl.drop([
            "min_totalPrice",
            "min_legs0_duration",
            "min_legs1_duration",
            "min_total_travel_time", 
            "pricingInfo_passengerCount",
            "corporateTariffCode"
        ])

        self.trainDFPl = self.trainDFPl.with_columns(
            ((pl.col("totalPrice") + 1)/(pl.col("legs0_duration").fill_null(0) + pl.col("legs1_duration").fill_null(0) + 1)).alias("priceDuration_TradeOff")
        )        
        gc.collect()
        print("Done others_and_duration")

    def arrival_and_monetory_adder(self): 
        cols = ['legs0_arrivalAt', 'legs0_departureAt', 'legs1_arrivalAt', 'legs1_departureAt']
        for x in cols:
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col(x).str.strip_chars().str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S", strict=False).alias(f"{x}_parsed")
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col(f"{x}_parsed").dt.hour().fill_null(0).alias(f"{x}_hour"),
                pl.col(f"{x}_parsed").dt.minute().fill_null(0).alias(f"{x}_minute"),
                pl.col(f"{x}_parsed").dt.weekday().fill_null(0).alias(f"{x}_weekday"),  # added weekday here
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col(f"{x}_hour") + (pl.col(f"{x}_minute") / 60)).alias(f"{x}_decimal_hour")
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                (((pl.col(f"{x}_hour") * 3600 + pl.col(f"{x}_minute") * 60) / 86400) * 360).alias(f"{x}_angle_deg")
            ])

            # Add cyclical encoding for weekday (0=Mon, 6=Sun)
            self.trainDFPl = self.trainDFPl.with_columns([
                (2 * np.pi * pl.col(f"{x}_weekday") / 7).map_elements(np.sin, return_dtype=pl.Float64).alias(f"{x}_weekday_sin"),
                (2 * np.pi * pl.col(f"{x}_weekday") / 7).map_elements(np.cos, return_dtype=pl.Float64).alias(f"{x}_weekday_cos"),
            ])
            
            #  Red-eye indicator (late night/early morning)
            self.trainDFPl[f"{x}_is_redeye"] = (
                (self.trainDFPl[f"{x}_hour"] >= 23) | (self.trainDFPl[f"{x}_hour"] < 6)
            ).astype(int)

            # Drop intermediates
            self.trainDFPl = self.trainDFPl.drop([f"{x}_parsed", f"{x}_hour", f"{x}_minute"])

        del cols
        gc.collect()

        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("miniRules0_monetaryAmount") / pl.col("totalPrice")).alias("miniRules0_monetaryAmount_ratio"),
            (pl.col("miniRules1_monetaryAmount") / pl.col("totalPrice")).alias("miniRules1_monetaryAmount_ratio"),
        ])
        
        self.trainDFPl = self.trainDFPl.drop([
            "miniRules0_monetaryAmount",
            "miniRules1_monetaryAmount",
            "legs0_arrivalAt",
            "legs0_departureAt",
            "legs1_arrivalAt",
            "legs1_departureAt",
            "legs0_duration",
            "legs1_duration",
        ])
        gc.collect()
        print("Done arrival_and_monetory_adder")


    def add_interaction_features(self):
        # Example: interaction between isVip and free_exchange
        if "isVip" in self.trainDFPl.columns and "free_exchange" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("isVip") * pl.col("free_exchange")).alias("vip_free_exchange_interaction")
            ])
        gc.collect()

    def returnProcessedDF(self):
        self.add_price_rank_features()
        self.add_min_stopage_per_trip()
        self.ff_flyer_bin_converter()
        self.hour_min_and_tax_converter()
        self.others_and_duration()
        self.arrival_and_monetory_adder()
        self.add_interaction_features()
        return self.trainDFPl


# Now train and work for XGB boosts

In [4]:
trainDFPl = PLDFPorcessor(trainPath).returnProcessedDF()
display(trainDFPl.head())
display(trainDFPl.to_pandas().head(10))

trainDFPl.write_parquet("/kaggle/working/processesdf.parquet")

trainDFPl = trainDFPl.drop([
    "Id", 
    #"legs0_arrivalAt", 
    #"legs0_departureAt",
    #"legs1_arrivalAt", 
    #"legs1_departureAt",
]).sort("ranker_id")


ranker_ids = trainDFPl["ranker_id"].unique().to_numpy()

# split DF based on ranker_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.01, random_state=42)
train_idx, valid_idx = next(gss.split(ranker_ids, groups=ranker_ids))

/tmp/ipykernel_36/2476130610.py:86: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().over("ranker_id").alias("group_count"),


Done ff_flyer_bin_converter!
Done hour_min_and_tax_converter
Done others_and_duration
Done arrival_and_monetory_adder


Id,bySelf,companyID,nationality,isAccess3D,isVip,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_seatsAvailable,legs1_segments0_baggageAllowance_quantity,legs1_segments0_baggageAllowance_weightMeasurementType,legs1_segments0_cabinClass,legs1_segments0_seatsAvailable,miniRules0_statusInfos,miniRules1_statusInfos,pricingInfo_isAccessTP,profileId,ranker_id,searchRoute,sex,taxes,totalPrice,selected,price_rank,duration_rank,price_pct_rank,log_price,has_corporate_tariff,minStopagePerRankderID,free_exchange,frequentFlyer_marketingCarrier_match,numberof_same_frequentFlyter_operator,total_travel_stopage,minimum_travel_segment,total_travel_time_in_minutes,tax_percentage,…,legs0_duration_rank_std,legs1_duration_rank_min,legs1_duration_rank_max,legs1_duration_rank_mean,legs1_duration_rank_std,total_travel_time_in_minutes_duration_rank_min,total_travel_time_in_minutes_duration_rank_max,total_travel_time_in_minutes_duration_rank_mean,total_travel_time_in_minutes_duration_rank_std,totalPrice_diff,legs0Duration_diff,legs1Duration_diff,totalDuration_diff,priceDuration_TradeOff,legs0_arrivalAt_decimal_hour,legs0_arrivalAt_sinDegree,legs0_arrivalAt_cosDegree,legs0_arrivalAt_weekday_sin,legs0_arrivalAt_weekday_cos,legs0_departureAt_decimal_hour,legs0_departureAt_sinDegree,legs0_departureAt_cosDegree,legs0_departureAt_weekday_sin,legs0_departureAt_weekday_cos,legs1_arrivalAt_decimal_hour,legs1_arrivalAt_sinDegree,legs1_arrivalAt_cosDegree,legs1_arrivalAt_weekday_sin,legs1_arrivalAt_weekday_cos,legs1_departureAt_decimal_hour,legs1_departureAt_sinDegree,legs1_departureAt_cosDegree,legs1_departureAt_weekday_sin,legs1_departureAt_weekday_cos,miniRules0_monetaryAmount_ratio,miniRules1_monetaryAmount_ratio,vip_free_exchange_interaction
i64,i8,i64,i64,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,u32,i8,f64,f64,i64,u32,u32,f64,f64,i8,i32,i8,i32,i32,i8,i8,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
0,1,57323,36,0,0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,null,null,1.0,2087645,"""98ce0dabf6964640b63079fbafd42c…",4,1,370.0,16884.0,1,1,1,0.0,9.734181,0,1,null,0,0,2,2,315.0,0.021914,…,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,0.0,0.0,0.0,0.0,53.433544,16.333333,-0.550481,0.834848,-0.781831,0.62349,15.666667,-0.739239,0.673443,-0.781831,0.62349,14.333333,-0.894154,0.447759,0.974928,-0.222521,9.75,0.713047,-0.701117,0.974928,-0.222521,null,null,null
1,1,57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,2087645,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,51125.0,0,2,2,0.041667,10.842048,1,0,0,4,8,4,2,950.0,0.043814,…,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,34241.0,285.0,350.0,635.0,53.760252,14.833333,-0.893894,0.448279,-0.781831,0.62349,9.416667,0.708956,-0.705253,-0.781831,0.62349,8.5,0.865734,-0.500504,0.433884,-0.900969,22.083333,0.839778,0.54293,0.974928,-0.222521,0.044988,0.06846,0
2,1,57323,36,0,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,2087645,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,53695.0,0,3,2,0.083333,10.891094,0,0,0,4,8,4,2,950.0,0.041717,…,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,36811.0,285.0,350.0,635.0,56.462671,14.833333,-0.893894,0.448279,-0.781831,0.62349,9.416667,0.708956,-0.705253,-0.781831,0.62349,8.5,0.865734,-0.500504,0.433884,-0.900969,22.083333,0.839778,0.54293,0.974928,-0.222521,0.042835,0.065183,0
3,1,57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,2087645,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,81880.0,0,4,2,0.125,11.313022,1,0,1,4,8,4,2,950.0,0.027357,…,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,64996.0,285.0,350.0,635.0,86.099895,14.833333,-0.893894,0.448279,-0.781831,0.62349,9.416667,0.708956,-0.705253,-0.781831,0.62349,8.5,0.865734,-0.500504,0.433884,-0.900969,22.083333,0.839

,Id,bySelf,companyID,nationality,isAccess3D,isVip,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_seatsAvailable,legs1_segments0_baggageAllowance_quantity,legs1_segments0_baggageAllowance_weightMeasurementType,legs1_segments0_cabinClass,legs1_segments0_seatsAvailable,miniRules0_statusInfos,miniRules1_statusInfos,pricingInfo_isAccessTP,profileId,ranker_id,searchRoute,sex,taxes,totalPrice,selected,price_rank,duration_rank,price_pct_rank,log_price,has_corporate_tariff,minStopagePerRankderID,free_exchange,frequentFlyer_marketingCarrier_match,numberof_same_frequentFlyter_operator,total_travel_stopage,minimum_travel_segment,total_travel_time_in_minutes,tax_percentage,legs0_duration_rank_min,legs0_duration_rank_max,legs0_duration_rank_mean,legs0_duration_rank_std,legs1_duration_rank_min,legs1_duration_rank_max,legs1_duration_rank_mean,legs1_duration_rank_std,total_travel_time_in_minutes_duration_rank_min,total_travel_time_in_minutes_duration_rank_max,total_travel_time_in_minutes_duration_rank_mean,total_travel_time_in_minutes_duration_rank_std,totalPrice_diff,legs0Duration_diff,legs1Duration_diff,totalDuration_diff,priceDuration_TradeOff,legs0_arrivalAt_decimal_hour,legs0_arrivalAt_sinDegree,legs0_arrivalAt_cosDegree,legs0_arrivalAt_weekday_sin,legs0_arrivalAt_weekday_cos,legs0_departureAt_decimal_hour,legs0_departureAt_sinDegree,legs0_departureAt_cosDegree,legs0_departureAt_weekday_sin,legs0_departureAt_weekday_cos,legs1_arrivalAt_decimal_hour,legs1_arrivalAt_sinDegree,legs1_arrivalAt_cosDegree,legs1_arrivalAt_weekday_sin,legs1_arrivalAt_weekday_cos,legs1_departureAt_decimal_hour,legs1_departureAt_sinDegree,legs1_departureAt_cosDegree,legs1_departureAt_weekday_sin,legs1_departureAt_weekday_cos,miniRules0_monetaryAmount_ratio,miniRules1_monetaryAmount_ratio,vip_free_exchange_interaction
0,0,1,57323,36,0,0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,NaN,NaN,1.0,2087645,98ce0dabf6964640b63079fbafd42cbe,4,1,370.0,16884.0,1,1,1,0.000000,9.734181,0,1,NaN,0,0,2,2,315.0,0.021914,160.0,595.0,505.6,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,0.0,0.0,0.0,0.0,53.433544,16.333333,-0.550481,0.834848,-0.781831,0.62349,15.666667,-0.739239,0.673443,-0.781831,0.62349,14.333333,-0.894154,0.447759,0.974928,-0.222521,9.750000,0.713047,-0.701117,0.974928,-0.222521,NaN,NaN,NaN
1,1,1,57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,2087645,98ce0dabf6964640b63079fbafd42cbe,4,1,2240.0,51125.0,0,2,2,0.041667,10.842048,1,0,0.0,4,8,4,2,950.0,0.043814,160.0,595.0,505.6,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,34241.0,285.0,350.0,635.0,53.760252,14.833333,-0.893894,0.448279,-0.781831,0.62349,9.416667,0.708956,-0.705253,-0.781831,0.62349,8.500000,0.865734,-0.500504,0.433884,-0.900969,22.083333,0.839778,0.542930,0.974928,-0.222521,0.044988,0.068460,0.0
2,2,1,57323,36,0,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,2087645,98ce0dabf6964640b63079fbafd42cbe,4,1,2240.0,53695.0,0,3,2,0.083333,10.891094,0,0,0.0,4,8,4,2,950.0,0.041717,160.0,595.0,505.6,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,36811.0,285.0,350.0,635.0,56.462671,14.833333,-0.893894,0.448279,-0.781831,0.62349,9.416667,0.708956,-0.705253,-0.781831,0.62349,8.500000,0.865734,-0.500504,0.433884,-0.900969,22.083333,0.839778,0.542930,0.974928,-0.222521,0.042835,0.065183,0.0
3,3,1,57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,2087645,98ce0dabf6964640b63079fbafd42cbe,4,1,2240.0,81880.0,0,4,2,0.125000,11.313022,1,0,1.0,4,8,4,2,950.0,0.027357,160.0,595.0,505.6,103.966341,155.0,755.0,619.0,141.06736,315.0,1350.0,1124.6,211.255769,64996.0,285.0,350.0,635.0,86.099895,14.833333,-0.893894,0.448279,-0.781831,0.62349,9.416667,0.708956,-0.705253,-0.781831,0.62349,8.500000,0.865734,-0.500504,0.433884,-0.900969,22.083333,0.839778,0.542930,0.974928,-0.222521,0.000000,0.000000,0.0
4,4,1,57323,36,0,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,208764

In [5]:
train_feature_cols = [col for col in trainDFPl.columns if col not in ("selected", "ranker_id")]

train_rankers = ranker_ids[train_idx]
valid_rankers = ranker_ids[valid_idx]

train_df = trainDFPl.filter(pl.col("ranker_id").is_in(train_rankers)).sort("ranker_id")
# train_df = trainDFPl.sort("ranker_id")
valid_df = trainDFPl.filter(pl.col("ranker_id").is_in(valid_rankers)).sort("ranker_id")

display(trainDFPl.head())

gss, train_idx, valid_idx , trainDFPl, train_rankers, valid_rankers = None, None, None, None, None, None
del gss, train_idx, valid_idx , trainDFPl, train_rankers, valid_rankers
pa.default_memory_pool().release_unused()
gc.collect()

bySelf,companyID,nationality,isAccess3D,isVip,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_seatsAvailable,legs1_segments0_baggageAllowance_quantity,legs1_segments0_baggageAllowance_weightMeasurementType,legs1_segments0_cabinClass,legs1_segments0_seatsAvailable,miniRules0_statusInfos,miniRules1_statusInfos,pricingInfo_isAccessTP,profileId,ranker_id,searchRoute,sex,taxes,totalPrice,selected,price_rank,duration_rank,price_pct_rank,log_price,has_corporate_tariff,minStopagePerRankderID,free_exchange,frequentFlyer_marketingCarrier_match,numberof_same_frequentFlyter_operator,total_travel_stopage,minimum_travel_segment,total_travel_time_in_minutes,tax_percentage,legs0_duration_rank_min,…,legs0_duration_rank_std,legs1_duration_rank_min,legs1_duration_rank_max,legs1_duration_rank_mean,legs1_duration_rank_std,total_travel_time_in_minutes_duration_rank_min,total_travel_time_in_minutes_duration_rank_max,total_travel_time_in_minutes_duration_rank_mean,total_travel_time_in_minutes_duration_rank_std,totalPrice_diff,legs0Duration_diff,legs1Duration_diff,totalDuration_diff,priceDuration_TradeOff,legs0_arrivalAt_decimal_hour,legs0_arrivalAt_sinDegree,legs0_arrivalAt_cosDegree,legs0_arrivalAt_weekday_sin,legs0_arrivalAt_weekday_cos,legs0_departureAt_decimal_hour,legs0_departureAt_sinDegree,legs0_departureAt_cosDegree,legs0_departureAt_weekday_sin,legs0_departureAt_weekday_cos,legs1_arrivalAt_decimal_hour,legs1_arrivalAt_sinDegree,legs1_arrivalAt_cosDegree,legs1_arrivalAt_weekday_sin,legs1_arrivalAt_weekday_cos,legs1_departureAt_decimal_hour,legs1_departureAt_sinDegree,legs1_departureAt_cosDegree,legs1_departureAt_weekday_sin,legs1_departureAt_weekday_cos,miniRules0_monetaryAmount_ratio,miniRules1_monetaryAmount_ratio,vip_free_exchange_interaction
i8,i64,i64,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,u32,i8,f64,f64,i64,u32,u32,f64,f64,i8,i32,i8,i32,i32,i8,i8,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
1,41022,36,0,0,0.0,0.0,1.0,6.0,null,null,null,null,1.0,0.0,1.0,421977,"""00004b95a9e446b586c5e9b7297144…",2,0,2830.0,9040.0,0,1,5,0.0,9.109525,0,0,0,0,0,2,1,1385.0,0.313053,120.0,…,477.365538,0.0,0.0,0.0,0.0,120.0,1385.0,472.0,477.365538,0.0,1265.0,0.0,1265.0,6.523088,20.666667,0.459166,0.88835,0.433884,-0.900969,21.583333,0.671074,0.741391,0.974928,-0.222521,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.327434,0.0,0
1,41022,36,0,0,1.0,0.0,1.0,6.0,null,null,null,null,1.0,0.0,1.0,421977,"""00004b95a9e446b586c5e9b7297144…",2,0,2830.0,13740.0,0,3,5,0.142857,9.528139,0,0,0,0,0,2,1,1385.0,0.205968,120.0,…,477.365538,0.0,0.0,0.0,0.0,120.0,1385.0,472.0,477.365538,4700.0,1265.0,0.0,1265.0,9.914141,20.666667,0.459166,0.88835,0.433884,-0.900969,21.583333,0.671074,0.741391,0.974928,-0.222521,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.291121,0.0,0
1,41022,36,0,0,1.0,0.0,1.0,6.0,null,null,null,null,1.0,1.0,1.0,421977,"""00004b95a9e446b586c5e9b7297144…",2,0,2830.0,20440.0,0,6,5,0.357143,9.925298,0,0,0,0,0,2,1,1385.0,0.138454,120.0,…,477.365538,0.0,0.0,0.0,0.0,120.0,1385.0,472.0,477.365538,11400.0,1265.0,0.0,1265.0,14.748196,20.666667,0.459166,0.88835,0.433884,-0.900969,21.583333,0.671074,0.741391,0.974928,-0.222521,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.195695,0.195695,0
1,41022,36,1,0,0.0,0.0,1.0,1.0,null,null,null,null,1.0,0.0,1.0,421977,"""00004b95a9e446b586c5e9b7297144…",2,0,577.0,21897.0,0,7,1,0.428571,9.994151,1,1,0,0,0,1,1,120.0,0.026351,120.0,…,477.365538,0.0,0.0,0.0,0.0,120.0,1385.0,472.0,477.365538,12857.0,0.0,0.0,0.0,180.975207,2.166667,0.505532,0.862808,0.974928,-0.222521,0.166667,0.006399,0.99998,0.974928,-0.222521,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.127871,0.0,0
1,41022,36,1,0,1.0,0.0,1.0,1.0,null,null,null,null,1.0,1.0,1.0,421977,"""00004b95a9e446b586c5e9b7297144…",2,0,577.0,24057.0,1,8,1,0.5,10.088223,1,1,0,0,0,1,1,120.0,

0

# Create groups and prepare model trainer for xgb ranker: ===================

In [6]:
# create groups for XGB ranker and LGB ranker
group_train_xgb = (
    train_df.group_by("ranker_id").len()
    .sort("ranker_id")["len"]
    .to_numpy()
    .astype("uint32")
)
group_valid_xgb = (
    valid_df.group_by("ranker_id").len()
    .sort("ranker_id")["len"]
    .to_numpy()
    .astype("uint32")
)

pa.default_memory_pool().release_unused()
gc.collect()

0

In [15]:
train_df_select = train_df["selected"].to_numpy()
valid_df_select = valid_df["selected"].to_numpy()

train_df = train_df.drop(["selected", "ranker_id"]).to_numpy()
valid_df = valid_df.drop(["selected", "ranker_id"]).to_numpy()

pa.default_memory_pool().release_unused()
gc.collect()

225

In [16]:
dtrain_xgb = xgb.QuantileDMatrix(
    train_df,
    label=train_df_select,
    group=group_train_xgb
)

train_df = None
del train_df
pa.default_memory_pool().release_unused()
gc.collect()

dvalid_xgb = xgb.QuantileDMatrix(
    valid_df,
    label=valid_df_select,
    group=group_valid_xgb
)

valid_df = None 
del valid_df
pa.default_memory_pool().release_unused()
gc.collect()

11

# DMatrix preparation for train
dtrain_xgb = xgb.DMatrix(   
    train_df , 
    label = train_df_select
) 
dtrain_xgb.set_group(group_train_xgb) 
 
train_df = None 
del train_df
pa.default_memory_pool().release_unused() 
gc.collect() 
 

# DMatrix preparation for valid
dvalid_xgb = xgb.DMatrix( 
    valid_df,  
    label = valid_df_select 
) 
dvalid_xgb.set_group(group_valid_xgb) 

valid_df = None  
del valid_df 
pa.default_memory_pool().release_unused()
gc.collect() 

### train xgb ranker model

In [17]:
params = {
    'objective': 'rank:ndcg',
    "eval_metric": ["ndcg@3", "ndcg@10"],
    "learning_rate": 0.022641389657079056,
    "max_depth": 0,
    "min_child_weight": 2 , # 2
    "subsample": 0.8842234913702768,
    "colsample_bytree": 0.45840689146263086,
    "gamma": 3.3084297630544888,
    "lambda": 6.952586917313028,
    "alpha": 0.6395254133055179,
    'seed': 42,
    'n_jobs': -1,
    'device': 'cuda'
}


trainedXGBoostsModel = xgb.train(
    params = params,
    dtrain = dtrain_xgb,
    num_boost_round = 500,
    evals=[(dtrain_xgb, "train"), (dvalid_xgb, "valid")],
    # early_stopping_rounds=100,
    verbose_eval=10
)

[0]	train-ndcg@3:0.35301	train-ndcg@10:0.43625	valid-ndcg@3:0.35516	valid-ndcg@10:0.43965
[10]	train-ndcg@3:0.84650	train-ndcg@10:0.86999	valid-ndcg@3:0.49966	valid-ndcg@10:0.56661
[20]	train-ndcg@3:0.90360	train-ndcg@10:0.91684	valid-ndcg@3:0.52493	valid-ndcg@10:0.59241
[30]	train-ndcg@3:0.91984	train-ndcg@10:0.93062	valid-ndcg@3:0.53403	valid-ndcg@10:0.60165
[40]	train-ndcg@3:0.92257	train-ndcg@10:0.93314	valid-ndcg@3:0.54196	valid-ndcg@10:0.60793
[50]	train-ndcg@3:0.92770	train-ndcg@10:0.93747	valid-ndcg@3:0.54741	valid-ndcg@10:0.61354
[60]	train-ndcg@3:0.92988	train-ndcg@10:0.93959	valid-ndcg@3:0.54881	valid-ndcg@10:0.61253
[70]	train-ndcg@3:0.93317	train-ndcg@10:0.94236	valid-ndcg@3:0.55355	valid-ndcg@10:0.61659
[80]	train-ndcg@3:0.93517	train-ndcg@10:0.94415	valid-ndcg@3:0.55347	valid-ndcg@10:0.61620
[90]	train-ndcg@3:0.93589	train-ndcg@10:0.94470	valid-ndcg@3:0.55337	valid-ndcg@10:0.61674
[100]	train-ndcg@3:0.93732	train-ndcg@10:0.94603	valid-ndcg@3:0.55479	valid-ndcg@10:0.61634

In [18]:
dtrain_xgb, dvalid_xgb, group_train_xgb, group_valid_xgb = None , None, None , None
del dtrain_xgb, dvalid_xgb, group_train_xgb, group_valid_xgb

pa.default_memory_pool().release_unused()
gc.collect()

37

## XGB Ranker: check the important cols according to their weights

In [19]:
gain_importance = trainedXGBoostsModel.get_score(importance_type='gain')
colRename_map = {f"f{idx}": col for idx, col in enumerate(train_feature_cols)}

gain_importance_df = pd.DataFrame(
    list(gain_importance.items()),
    columns=['feature', 'importance']
)
gain_importance_df['feature'] = gain_importance_df['feature'].map(colRename_map)
gain_importance_df = gain_importance_df.sort_values(by='importance', ascending=False)
gain_importance_df.reset_index(drop=True, inplace=True)

print(len(gain_importance_df))
gain_importance_df

73


,feature,importance
0,minStopagePerRankderID,2422.526855
1,free_exchange,672.841064
2,legs0_segments0_cabinClass,202.314819
3,legs0_segments0_baggageAllowance_quantity,108.010910
4,totalDuration_diff,96.761047
5,legs0Duration_diff,53.276360
6,legs0_segments0_baggageAllowance_weightMeasure...,51.072002
7,pricingInfo_isAccessTP,41.928642
8,total_travel_stopage,33.092880
9,minimum_travel_segment,28.155750


In [ ]:
gain_importance, colRename_map , gain_importance_df = None , None , None 
del gain_importance, colRename_map , gain_importance_df
pa.default_memory_pool().release_unused()
gc.collect()

# ==================== Predict and Dataset task ====================

# Now work for test Dataset for xgb ranker: --------------------

In [ ]:
testDFPl = PLDFPorcessor(testPath).returnProcessedDF().sort("ranker_id")

testIds_xgb = testDFPl["Id"].to_numpy()
testRanker_ids_xgb = testDFPl["ranker_id"].to_numpy()

test_feature_cols = [col for col in testDFPl.columns if col not in ("Id", "ranker_id")]

In [ ]:
passToPredict_df = testDFPl.select(test_feature_cols).to_numpy()

group_test = (
    testDFPl.group_by("ranker_id").len()
    .sort("ranker_id")["len"]
    .to_numpy()
    .astype("uint32")
)

pa.default_memory_pool().release_unused()
gc.collect()

In [ ]:
dtest = xgb.DMatrix(passToPredict_df)
dtest.set_group(group_test)

pred_xgbRanker = trainedXGBoostsModel.predict(dtest)

dtest, passToPredict_df, group_test = None, None, None
del dtest, passToPredict_df, group_test
pa.default_memory_pool().release_unused()
gc.collect()

## create a csv file for xgb ranker

In [ ]:
xgbSubmissionDF = pl.DataFrame({
    "Id": testIds_xgb,
    "ranker_id": testRanker_ids_xgb,
    "selected_xgb": pred_xgbRanker
})

xgbSubmissionDF = xgbSubmissionDF.with_columns([
    pl.col("selected_xgb")
    .rank(method="ordinal", descending=True)
    .over("ranker_id")
    .alias("rank_xgb")
])


xgbSubmissionDF = xgbSubmissionDF.sort("Id", descending=False) # descending=False -> small to big
xgbSubmissionDF.head(10)

In [ ]:
xgbSubmissionDF = xgbSubmissionDF.drop([
    "selected_xgb"
]).rename({"rank_xgb": "selected"})

In [ ]:
xgbSubmissionDF.head(10)

In [ ]:
xgbSubmissionDF.write_csv(os.path.join(os.path.abspath("."), "submission.csv"))